# BwithU: make new 3D faces (free Colab GPU)

Turns portraits into LAM talking-head files for the BwithU app (one `.zip` per face).

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** The first run installs everything (about 30–40 minutes). Leave this tab open.
3. At the end your browser downloads **`bwithu_faces.zip`**. Give that file to Claude.

Uses only AI-generated portraits from the BwithU repo by default. To add your own, run the optional upload cell
(only photos of people who agreed: no celebrities or strangers).

If a cell fails, copy the last ~30 lines of its output to Claude.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
%%bash
# 2. Install LAM in its own Python 3.10 environment (about 15-25 minutes)
set -e
cd /content
echo "== system packages"
apt-get -qq update > /dev/null
apt-get -qq install -y libxi6 libxkbcommon0 libxrender1 libsm6 libgl1 libegl1 zip > /dev/null
echo "== python 3.10"
pip -q install uv
[ -d /content/py310 ] || uv venv -q --seed -p 3.10 /content/py310
source /content/py310/bin/activate
export UV_HTTP_TIMEOUT=300
[ -d LAM ] || git clone -q https://github.com/aigc3d/LAM.git
cd LAM
echo "== torch 2.5.1 + prebuilt pytorch3d"
uv pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
uv pip install -q xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121
uv pip install -q pytorch3d --no-index -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt251/download.html
echo "== LAM requirements"
pip -q install setuptools wheel ninja Cython
grep -v -E "pytorch3d|diff-gaussian|simple-knn|nvdiffrast" requirements.txt > /tmp/req.txt
pip -q install --no-build-isolation -r /tmp/req.txt
echo "== CUDA extensions (a few minutes)"
export TORCH_CUDA_ARCH_LIST="7.5" MAX_JOBS=4
[ -d /content/dgr ] || git clone -q --recursive https://github.com/ashawkey/diff-gaussian-rasterization /content/dgr
[ -d /content/sknn ] || git clone -q https://github.com/camenduru/simple-knn.git /content/sknn
pip -q install --no-build-isolation /content/dgr /content/sknn "nvdiffrast@git+https://github.com/ShenhanQian/nvdiffrast@backface-culling"
(cd external/landmark_detection/FaceBoxesV2/utils && sh make.sh > /dev/null)
python -c "import torch, pytorch3d, diff_gaussian_rasterization, simple_knn; print('OK torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
%%bash
# 3. Download LAM models, Blender, FBX SDK and the chat-avatar template (about 5-10 minutes)
set -e
source /content/py310/bin/activate
cd /content/LAM
if [ ! -f model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors ]; then
  huggingface-cli download 3DAIGC/LAM-assets --local-dir ./tmp > /dev/null
  tar -xf ./tmp/LAM_assets.tar && rm ./tmp/LAM_assets.tar
  tar -xf ./tmp/thirdparty_models.tar && rm -r ./tmp/
  huggingface-cli download 3DAIGC/LAM-20K --local-dir ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/ > /dev/null
fi
if [ ! -x /content/blender/blender ]; then
  wget -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz -O /tmp/blender.tar.xz
  mkdir -p /content/blender && tar -xf /tmp/blender.tar.xz -C /content/blender --strip-components=1
fi
pip -q install https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl patool
if [ ! -d assets/sample_oac ]; then
  wget -q https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/sample_oac.tar -O /tmp/oac.tar
  tar -xf /tmp/oac.tar -C assets/
fi
ls model_zoo/lam_models/releases/lam/lam-20k/step_045500/ assets/sample_oac/
/content/blender/blender --version | head -1

In [ ]:
# 4. Get the BwithU portraits (AI-generated) from GitHub
import os, urllib.request
os.makedirs('/content/portraits', exist_ok=True)
for name in ['maya', 'elena', 'zara', 'aiko', 'leo', 'ravi']:
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/AmanYadav007/bwithu/main/avatars-src/{name}.jpg', f'/content/portraits/{name}.jpg')
print(sorted(os.listdir('/content/portraits')))

In [ ]:
# 4b. OPTIONAL: upload more portraits (front-facing, one face, good light). The file name becomes the face id,
# so use a short lowercase name like "nina.jpg". Skip this cell if you don't need it.
RUN_UPLOAD = False
if RUN_UPLOAD:
    from google.colab import files
    for name, data in files.upload().items():
        open(f'/content/portraits/{name.lower()}', 'wb').write(data)
    print(sorted(os.listdir('/content/portraits')))

In [ ]:
%%bash
# 5. Make the faces (about 1-3 minutes per portrait)
source /content/py310/bin/activate
cd /content/LAM
python - <<'EOF'
p = 'app_lam.py'
s = open(p).read()
hook = """
        # BwithU batch mode: export an OpenAvatarChat ZIP for every image in $BATCH_DIR, no UI.
        if os.environ.get(\"BATCH_DIR\"):
            video = sorted(glob(\"./assets/sample_motion/export/*/*.mp4\"))[0]
            images = sorted(glob(os.path.join(os.environ[\"BATCH_DIR\"], \"*.jp*g\")) + glob(os.path.join(os.environ[\"BATCH_DIR\"], \"*.png\")))
            for img in images:
                print(\"BATCH start\", img, flush=True)
                try:
                    out = core_fn(img, video, prepare_working_dir(), True)
                    print(\"BATCH done\", img, \"->\", out[-1], flush=True)
                except Exception as e:
                    import traceback; traceback.print_exc()
                    print(\"BATCH FAILED\", img, repr(e), flush=True)
            return
"""
if 'BATCH_DIR' not in s:
    assert s.count('        demo.queue()\n') == 1, 'app_lam.py changed upstream; tell Claude'
    s = s.replace('        demo.queue()\n', hook + '        demo.queue()\n')
    open(p, 'w').write(s)
print('patched')
EOF
BATCH_DIR=/content/portraits python app_lam.py --blender_path /content/blender/blender > /content/run.log 2>&1
echo "exit code $?"
grep BATCH /content/run.log
ls -la output/open_avatar_chat/ 2>/dev/null || (echo "no faces made; last log lines:"; tail -40 /content/run.log)

In [ ]:
# 6. Download all faces as bwithu_faces.zip
import glob, shutil
from google.colab import files
zips = sorted(glob.glob('/content/LAM/output/open_avatar_chat/*.zip'))
print(zips)
assert zips, 'No faces were made: see the output of step 5'
shutil.make_archive('/content/bwithu_faces', 'zip', '/content/LAM/output/open_avatar_chat')
files.download('/content/bwithu_faces.zip')